## 量化FP32模型并导出单个测试图片计算过程中的所有INT8数据
1. **其中Scale参数来源于训练集采样** 

In [2]:
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np

# ==========================================
# 0. 定义手动展平函数
# ==========================================
def my_flatten(x):
    """
    手动实现展平操作，严格按照 C/C++ 的行优先 (Row-Major) 逻辑。
    """
    B, C, H, W = x.shape
    flattened_memory = []
    for b in range(B):
        for c in range(C):
            for h in range(H):
                for w in range(W):
                    val = x[b, c, h, w].item() 
                    flattened_memory.append(val)
    return torch.tensor([flattened_memory], dtype=x.dtype, device=x.device)

# ==========================================
# 1. 完善后的全网定义
# ==========================================
class HardwareFriendlyCNN(torch.nn.Module):
    def __init__(self):
        super(HardwareFriendlyCNN, self).__init__()
        self.conv1 = torch.nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = torch.nn.MaxPool2d(2, 2)
        self.conv2 = torch.nn.Conv2d(16, 32, kernel_size=3, padding=1)
        # 注意这里的 32*8*8，说明经历了两次 2x2 的池化 (32 -> 16 -> 8)
        self.fc = torch.nn.Linear(32 * 8 * 8, 10)
        self.relu = torch.nn.ReLU()
        
    def forward(self, x):
        # 这个前向传播严格定义了数据的物理流向
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1) # 训练用的 flatten，也是用于展平的， -1表示由python自动推断这一维度的大小。
        x = self.fc(x)
        return x

# ==========================================
# 2. 量化核心工具函数
# ==========================================
def sym_quantize(tensor):
    """对称量化：强制 Zero Point = 0"""
    max_abs = tensor.abs().max().item()
    scale = max_abs / 127.0 if max_abs > 0 else 1.0
    q_tensor = torch.round(tensor / scale)
    q_tensor = torch.clamp(q_tensor, -127, 127).to(torch.int8)
    return q_tensor, scale

def export_c_array(tensor, name, file_handle, c_type="int8_t"):
    """导出为一维 C 数组"""
    flat_data = tensor.flatten().tolist() #将 PyTorch 底层的 C++ 张量对象，转换成纯粹的 Python 列表，用于for循环
    file_handle.write(f"const {c_type} {name}[{len(flat_data)}] = {{\n    ") #{{ 的转义机制
    for i, val in enumerate(flat_data):#enumerate 会同时返回数据的当前索引 i 和具体的数值 val
        file_handle.write(f"{val}, ")
        if (i + 1) % 16 == 0:
            file_handle.write("\n    ")
    file_handle.write("\n};\n\n")

def calc_multiplier_shift(s_in, s_w, s_out, shift=16):
    """计算硬件所需的 Multiplier，Shift 固定为 16 以简化硬件设计"""# multiplier是用于重新量化，Shift是其被放大的位数
    M = (s_in * s_w) / s_out
    multiplier = int(round(M * (1 << shift)))
    return multiplier, shift

# ==========================================
# 3. 主流程：计算 Scale，生成黄金模型
# ==========================================
def main():
    print(">>> 步骤 1/4: 加载模型与数据集...")
    model = HardwareFriendlyCNN()
    model.load_state_dict(torch.load('my_hardware_cnn_fp32_69p.pth', weights_only=True)) #防御机制,防止执行脚本
    model.eval()
    #Compose是流水线打包盒
    transform = transforms.Compose([transforms.ToTensor()])#原始图片的像素值是 0 到 255 的 uint8 整数，ToTensor() 会自动将这些整数除以 255
    
    # 【规范化升级】使用训练集作为校准集，彻底规避数据泄露！
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    # 取前 1000 张训练集图片作为校准数据足够了
    calib_loader = torch.utils.data.DataLoader(torch.utils.data.Subset(trainset, range(1000)), batch_size=100, shuffle=False)
    
    # 加载测试集，只为了提取我们的“黄金测试向量 1”
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    
    # 【锁定目标】提取测试集 Index = 1 的图片
    TARGET_INDEX = 1
    target_img_fp32, target_label = testset[TARGET_INDEX]
    target_img_fp32 = target_img_fp32.unsqueeze(0) #维度扩充，从(3, 32, 32) 变成 (1, 3, 32, 32)
    print(f"提取黄金测试图片成功 (Index={TARGET_INDEX})！真实标签: {target_label}")

    print("\n>>> 步骤 2/4: 执行严格的 PTQ 静态校准 (基于训练集)...")
    global_max_in, global_max_out1, global_max_out2, global_max_fc = 0.0, 0.0, 0.0, 0.0

    with torch.no_grad():#使用F中的各种标准函数：精妙之处在于“拦截数据流”
        for images, _ in calib_loader:
            global_max_in = max(global_max_in, images.abs().max().item())
            out1 = F.conv2d(images, model.conv1.weight, model.conv1.bias, padding=1)
            global_max_out1 = max(global_max_out1, out1.abs().max().item())
            pool1 = F.max_pool2d(F.relu(out1), 2)
            out2 = F.conv2d(pool1, model.conv2.weight, model.conv2.bias, padding=1)
            global_max_out2 = max(global_max_out2, out2.abs().max().item())
            pool2 = F.max_pool2d(F.relu(out2), 2)
            flat = pool2.view(pool2.size(0), -1)
            fc_out = F.linear(flat, model.fc.weight, model.fc.bias)
            global_max_fc = max(global_max_fc, fc_out.abs().max().item())

    # 更加硬件友好的安全写法：如果最大值为 0（极罕见），强制设为 1.0 防止 M 爆炸
    s_in = global_max_in / 127.0 if global_max_in > 0 else 1.0
    s_out1 = global_max_out1 / 127.0 if global_max_out1 > 0 else 1.0
    s_out2 = global_max_out2 / 127.0 if global_max_out2 > 0 else 1.0
    s_out_fc = global_max_fc / 127.0 if global_max_fc > 0 else 1.0

    print("\n>>> 步骤 3/4: 固化量化参数与权重...")
    # Layer 1
    w1_q, s_w1 = sym_quantize(model.conv1.weight)
    b1_q = torch.round(model.conv1.bias / (s_in * s_w1)).to(torch.int32)
    mult1, shift1 = calc_multiplier_shift(s_in, s_w1, s_out1)
    
    # Layer 2
    w2_q, s_w2 = sym_quantize(model.conv2.weight)
    b2_q = torch.round(model.conv2.bias / (s_out1 * s_w2)).to(torch.int32)
    mult2, shift2 = calc_multiplier_shift(s_out1, s_w2, s_out2)
    
    # Layer 3 (FC)
    wfc_q, s_wfc = sym_quantize(model.fc.weight)
    bfc_q = torch.round(model.fc.bias / (s_out2 * s_wfc)).to(torch.int32)
    mult_fc, shift_fc = calc_multiplier_shift(s_out2, s_wfc, s_out_fc)

    print("\n>>> 步骤 4/4: 为靶向目标 (Index=1) 生成 Bit-Accurate 黄金输出...")
    # 1. 严格使用全局 s_in 量化输入
    target_img_q = torch.round(target_img_fp32 / s_in)
    target_img_q = torch.clamp(target_img_q, -127, 127).to(torch.int8)

    # 2. 纯 INT8 前向推演
    mac1 = F.conv2d(target_img_q.float(), w1_q.float(), bias=b1_q.float(), padding=1).to(torch.int32)
    q_conv1_out = torch.clamp((mac1 * mult1) >> shift1, -127, 127).to(torch.int8)
    q_pool1_out = F.max_pool2d(F.relu(q_conv1_out.float()), 2).to(torch.int8)

    mac2 = F.conv2d(q_pool1_out.float(), w2_q.float(), bias=b2_q.float(), padding=1).to(torch.int32)
    q_conv2_out = torch.clamp((mac2 * mult2) >> shift2, -127, 127).to(torch.int8)
    q_pool2_out = F.max_pool2d(F.relu(q_conv2_out.float()), 2).to(torch.int8)

    q_flat_in = my_flatten(q_pool2_out)
    mac_fc = F.linear(q_flat_in.float(), wfc_q.float(), bias=bfc_q.float()).to(torch.int32)
    q_fc_out = torch.clamp((mac_fc * mult_fc) >> shift_fc, -127, 127).to(torch.int8)

    predicted_label = torch.argmax(q_fc_out, dim=1).item()
    print(f"🎯 INT8 黄金模型对 Target 的预测结果: {predicted_label} (对比真实标签: {target_label})")

    # 导出到 test_vector.h ...
    with open("test_vector.h", "w") as f:
        f.write("#ifndef TEST_VECTOR_H\n#define TEST_VECTOR_H\n\n#include <stdint.h>\n\n")
        f.write("// ================= 全局统一量化参数 =================\n")
        
        # 将 s_in 存为浮点常量 (如果预处理在单片机/PicoRV32上做，会用到它；如果全由PC处理好再传进去，则仅作记录)
        f.write(f"const float GLOBAL_S_IN = {s_in};\n\n")
        
        # 导出 Layer 1 量化参数
        f.write(f"const int32_t L1_MULT = {mult1};\n")
        f.write(f"const int32_t L1_SHIFT = {shift1};\n\n")
        
        # 导出 Layer 2 量化参数
        f.write(f"const int32_t L2_MULT = {mult2};\n")
        f.write(f"const int32_t L2_SHIFT = {shift2};\n\n")
        
        # 导出 FC 层量化参数
        f.write(f"const int32_t FC_MULT = {mult_fc};\n")
        f.write(f"const int32_t FC_SHIFT = {shift_fc};\n\n")
        
        export_c_array(target_img_q, "test_input_img", f, "int8_t")
        
        # 导出 Layer 1
        f.write(f"// ================= Conv1 阶段 =================\n")
        export_c_array(w1_q, "test_conv1_weight", f, "int8_t")
        export_c_array(b1_q, "test_conv1_bias", f, "int32_t")
        f.write(f"// 黄金对照组：第一层 Conv + ReLU + Pool 后的输出结果 (大小 16x16x16=4096)\n")
        export_c_array(q_pool1_out, "expected_pool1_output", f, "int8_t")
        
        # 导出 Layer 2
        f.write(f"// ================= Conv2 阶段 =================\n")
        export_c_array(w2_q, "test_conv2_weight", f, "int8_t")
        export_c_array(b2_q, "test_conv2_bias", f, "int32_t")
        f.write(f"// 黄金对照组：第二层 Conv + ReLU + Pool 后的输出结果 (大小 32x8x8=2048)\n")
        export_c_array(q_pool2_out, "expected_pool2_output", f, "int8_t")
        
        # 导出 Layer 3 (FC)
        f.write(f"// ================= FC 阶段 =================\n")
        export_c_array(wfc_q, "test_fc_weight", f, "int8_t")
        export_c_array(bfc_q, "test_fc_bias", f, "int32_t")
        f.write(f"// 黄金对照组：全连接层最终 10 分类输出\n")
        export_c_array(q_fc_out, "expected_fc_output", f, "int8_t")

        f.write("#endif // TEST_VECTOR_H\n")
    print("静态全局量化的黄金对账单导出完毕！")

if __name__ == "__main__":
    main()

>>> 步骤 1/4: 加载模型与数据集...
Files already downloaded and verified
Files already downloaded and verified
提取黄金测试图片成功 (Index=1)！真实标签: 8

>>> 步骤 2/4: 执行严格的 PTQ 静态校准 (基于训练集)...

>>> 步骤 3/4: 固化量化参数与权重...

>>> 步骤 4/4: 为靶向目标 (Index=1) 生成 Bit-Accurate 黄金输出...
🎯 INT8 黄金模型对 Target 的预测结果: 8 (对比真实标签: 8)
静态全局量化的黄金对账单导出完毕！
